Sarah Sullivan

Last Updated: October 12, 2025

Created: October 12, 2025

02_analysis_v3.ipynb

INPUT: output from 02_analysis_v2.do, 02_analysis_v2.dta

This script contains markdown annotated code for the cleaning and reshaping of...

    Fomby, Paula. 2024. The Panel Study of Income Dynamics Family Composition File User Guide:
    Early Release. Data release (release number), Ann Arbor, MI: Survey Research Center, Institute
    for Social Research, University of Michigan. (Open ICPSR DOI).

In [ ]:
'''
# Create conda environment env_py312
conda create -n env_py312 python=3.12
conda activate env_py312
conda install ipykernel

# Assign environment
python -m ipykernel install --user --name env_py312 --display-name "Python (env_py312)"

# Check assignment
import sys
print(sys.executable)

# Restart kernel if in wrong environment
'''


# import necessary libraries
import pandas as pd
import dask.dataframe as dd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

Create a dataframe df_child by collapsing df such that each row is a child-year (df['is_child'] == TRUE & df[yr] = a value between 1968 and 1992). The width of the data set will be comprised of values for all variables of each individual (discrete values of individual_id) living in each child's household (same family number) in each year (yr) the child appears in the data. Each variable name should be the original variable name + _i, where i = individual's id. Drop any variables that are empty for all children in all periods. 

In [ ]:
# Assign directories 
root = "/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid"

# Change working directory
os.chdir(root)

In [ ]:
# Load data
df = pd.read_csv(root + "/02_analysis_v2.csv")

In [24]:
# assign individual id's = family id * 1000 + family number

df['individual_id'] = df['fam'] * 1000 + df['fam_number']

df = df[['individual_id', 'fam', 'fam_number', 'role'] + [col for col in df.columns if col not in ['individual_id', 'fam', 'fam_number', 'role']]]

df.head()

df['is_child'] = (df['role'] == "Son or daughter")

In [ ]:
df_child = df

df_child["adult"] = (df_child['I1'] >= 18)  

df_child["hh_roster"] = df_child.groupby(['fam', 'yr'])['individual_id'].transform(lambda x: ','.join(map(str, x)))

adult_groups = (
    df[df['adult']]
    .groupby(['fam', 'yr'])['individual_id']
    .apply(lambda ids: ','.join(map(str, ids)))
    .reset_index(name='adult_roster')
)

# Merge back to main df (all rows get the household's adult_roster)
df_child = df_child.merge(adult_groups, on=['fam', 'yr'], how='left')
df_child['adult_roster'] = df_child['adult_roster'].fillna('')


,fam,yr,individual_id,adult_roster
0,1,1968,1001,"1001,1002,1003,1004"
1,1,1968,1002,"1001,1002,1003,1004"
2,1,1968,1003,"1001,1002,1003,1004"
3,1,1968,1004,"1001,1002,1003,1004"
4,1,1968,1030,"1001,1002,1003,1004"


In [ ]:
df_child['household_roster'] = df_child['household_roster'].where(df_child['is_child'] != False, 'none')
df_child['adult_roster'] = df_child['adult_roster'].where(df_child['is_child'] != False, 'none')

In [38]:
df_child = df_child.sort_values(['fam', 'individual_id', 'yr'])

# Function to compute plus_adult for each person in each family
def compute_plus_adult(group):
    group = group.sort_values('yr')
    prev_roster = None
    plus_adult = []
    for roster in group['adult_roster']:
        current_set = set(str(roster).split(',')) if roster and roster != 'none' else set()
        if prev_roster is None:
            plus_adult.append(0)
        else:
            # plus_adult = 1 if any new adult appears in current_set not in prev_roster
            plus_adult.append(int(len(current_set - prev_roster) > 0))
        prev_roster = current_set
    group['plus_adult'] = plus_adult
    return group

df_child = df_child.groupby(['fam', 'individual_id'], group_keys=False).apply(compute_plus_adult)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_8021/3798594321.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_child = df_child.groupby(['fam', 'individual_id'], group_keys=False).apply(compute_plus_adult)


In [39]:
# Function to compute minus_adult for each person in each family
def compute_minus_adult(group):
    group = group.sort_values('yr')
    prev_roster = None
    minus_adult = []
    for roster in group['adult_roster']:
        current_set = set(str(roster).split(',')) if roster and roster != 'none' else set()
        if prev_roster is None:
            minus_adult.append(0)
        else:
            # minus_adult = 1 if any adult in prev_roster is not in current_set
            minus_adult.append(int(len(prev_roster - current_set) > 0))
        prev_roster = current_set
    group['minus_adult'] = minus_adult
    return group

df_child = df_child.groupby(['fam', 'individual_id'], group_keys=False).apply(compute_minus_adult)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_8021/2088541234.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_child = df_child.groupby(['fam', 'individual_id'], group_keys=False).apply(compute_minus_adult)


In [40]:
df_child['plus_adult'] = df_child['plus_adult'].where(df_child['is_child'] != False, 'none')
df_child['minus_adult'] = df_child['minus_adult'].where(df_child['is_child'] != False, 'none')


In [ ]:
# Summarize the 'plus_adult' column in df_child
plus_adult_summary = df_child['plus_adult'].value_counts(dropna=False)

if yr == 1968:
    print(plus_adult_summary)

plus_adult
none    1475475
0        128011
1         57039
Name: count, dtype: int64


In [42]:
plus_adult_by_year = df_child.groupby('yr')['plus_adult'].value_counts(dropna=False).unstack(fill_value=0)
print(plus_adult_by_year)

plus_adult     0     1   none
yr                           
1968        7402     0  59019
1969        5574  1828  59019
1970        5339  2063  59019
1971        5129  2273  59019
1972        4810  2592  59019
1973        4707  2695  59019
1974        4818  2584  59019
1975        4660  2742  59019
1976        4671  2731  59019
1977        4755  2647  59019
1978        4826  2576  59019
1979        4586  2816  59019
1980        4642  2760  59019
1981        5006  2396  59019
1982        4737  2665  59019
1983        4829  2573  59019
1984        5072  2330  59019
1985        5018  2384  59019
1986        5306  2096  59019
1987        5543  1859  59019
1988        5281  2121  59019
1989        5368  2034  59019
1990        5282  2120  59019
1991        5467  1935  59019
1992        5183  2219  59019


In [43]:
minus_adult_yr = df_child.groupby('yr')['minus' '_adult'].value_counts(dropna=False).unstack(fill_value=0)
print(minus_adult_yr)

minus_adult     0     1   none
yr                            
1968         7402     0  59019
1969         7322    80  59019
1970         6893   509  59019
1971         6794   608  59019
1972         6805   597  59019
1973         6692   710  59019
1974         6634   768  59019
1975         6558   844  59019
1976         6300  1102  59019
1977         6397  1005  59019
1978         6386  1016  59019
1979         6289  1113  59019
1980         6193  1209  59019
1981         6052  1350  59019
1982         6169  1233  59019
1983         5979  1423  59019
1984         5815  1587  59019
1985         5909  1493  59019
1986         5581  1821  59019
1987         5787  1615  59019
1988         5861  1541  59019
1989         5750  1652  59019
1990         5916  1486  59019
1991         5839  1563  59019
1992         5769  1633  59019


In [44]:
df_child_small = pd.DataFrame(df_child)

In [ ]:
df_child_small.head()


for each yr t through max(t), for kid_number i through max(i), in family_number k through max(k), 

